# EECE693 — Historical Photo Restoration & Colorization

**4 Model Variants (Ablation Study):**
- **Model 1:** U-Net, grayscale → RGB, L1 loss (baseline)
- **Model 2:** U-Net, L → ab channels, L1 + L2 loss (Lab color space)
- **Model 3:** Model 2 + VGG perceptual loss (sharper colors)
- **Model 4:** Model 3 + reference-guided cross-attention (our contribution)

---

## 1. Setup — Mount Drive & Check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Verify data folder exists (update this path if your Drive folder is named differently)
import os
data_path = "/content/drive/MyDrive/Pictures for Project"  # must match DATA_ROOT in Cell 6
if os.path.exists(data_path):
    folders = os.listdir(data_path)
    print(f"Data found! Folders: {folders}")
else:
    print(f"ERROR: Data not found at {data_path}")
    print("Update DATA_ROOT in the config cell below to match your Drive path.")

## 1b. One-Time Image Resize (run once for faster training)

In [ ]:
# ============================================================
# ONE-TIME DATA PREP: Resize images to 256px max for faster Drive reads
# Run this cell ONCE, then set DATA_ROOT in Cell 6 to the resized folder.
# Grayscale images are filtered out during resize so they never appear in training.
# Safe to interrupt and resume — already-resized images are skipped.
# ============================================================
from PIL import Image
import os, glob, json

SOURCE    = "/content/drive/MyDrive/Pictures for Project"
DEST      = "/content/drive/MyDrive/Pictures for Project Resized"
MAX_SIZE  = 256
SUBFOLDERS = [
    "Arab and Lebanese Diaspora",
    "Baptism",
    "Studio",
    "Colored_from_alfredarchive",
]

def _is_grayscale(img, threshold=5.0):
    img_small = img.copy()
    img_small.thumbnail((16, 16))
    import numpy as np
    arr = np.array(img_small, dtype=np.float32)
    return (abs(arr[:,:,0] - arr[:,:,1]).mean() < threshold and
            abs(arr[:,:,0] - arr[:,:,2]).mean() < threshold)

import time as _t

# Count total images first for upfront estimate
total_to_process = 0
all_src_paths = []
for subfolder in SUBFOLDERS:
    src_folder = os.path.join(SOURCE, subfolder)
    if not os.path.exists(src_folder):
        continue
    for pat in ["**/*.jpg", "**/*.JPG", "**/*.jpeg", "**/*.JPEG"]:
        found = glob.glob(os.path.join(src_folder, pat), recursive=True)
        all_src_paths.extend([(p, subfolder) for p in found])
total_to_process = len(all_src_paths)
print(f"Found {total_to_process} images to process.")
print(f"Rough estimate: {total_to_process * 2 / 3600:.1f}–{total_to_process * 4 / 3600:.1f} hours (depends on Drive speed).")
print("Progress printed every 200 images. Safe to interrupt and resume.")

count = 0
skipped_gray = 0
skipped_existing = 0
start_time = _t.time()

for src_path, subfolder in all_src_paths:
    rel      = os.path.relpath(src_path, SOURCE)
    dst_path = os.path.join(DEST, rel)
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    if os.path.exists(dst_path):
        skipped_existing += 1
        continue
    try:
        img = Image.open(src_path).convert("RGB")
        if _is_grayscale(img):
            skipped_gray += 1
            continue
        img.thumbnail((MAX_SIZE, MAX_SIZE), Image.LANCZOS)
        img.save(dst_path, "JPEG", quality=90)
        count += 1
        processed = count + skipped_gray
        if processed % 200 == 0:
            elapsed = _t.time() - start_time
            rate = processed / max(elapsed, 1)
            remaining = (total_to_process - skipped_existing - processed) / max(rate, 0.001)
            print(f"  [{processed}/{total_to_process - skipped_existing}] Resized: {count} | Grayscale skipped: {skipped_gray} | ~{remaining/3600:.1f}h remaining")
    except Exception as e:
        print(f"  Skipped {os.path.basename(src_path)}: {e}")

total_time = _t.time() - start_time
print(f"Done! Resized: {count} | Skipped grayscale: {skipped_gray} | Already done: {skipped_existing}")
print(f"Total time: {total_time/3600:.1f}h")
print(f"Resized folder: {DEST}")
print("Next step: change DATA_ROOT in Cell 6 to:")
print(f'  DATA_ROOT = "{DEST}"')


## 2. Configuration

In [ ]:
import os

# === Wrapper so all other cells can do `import config` style access ===
class config:
    # ----- MODEL VARIANT (change this to train different models) -----
    MODEL_VARIANT = 1

    # ----- PATHS -----
    DATA_ROOT = "/content/drive/MyDrive/Pictures for Project"
    OUTPUT_DIR = "/content/drive/MyDrive/EECE693_outputs"
    CHECKPOINT_DIR = "/content/drive/MyDrive/EECE693_checkpoints"  # save to Drive so they persist

    # Training folders: color images only (we need ground-truth colors)
    TRAIN_SUBFOLDERS = [
        "Arab and Lebanese Diaspora",
        "Baptism",
        "Studio",
        "Colored_from_alfredarchive",
    ]

    # Test folders: B&W scans (no color ground truth)
    TEST_SUBFOLDERS = [
        "alfred archive/alfred moussa emile scan",
        "alfred archive/emil boulos scan 2013",
        "alfred archive/emil boulos b and white",
    ]

    # ----- IMAGE SETTINGS -----
    IMAGE_SIZE = 128
    IMAGE_CHANNELS = 1
    OUTPUT_CHANNELS = 3

    # ----- SYNTHETIC DEGRADATION -----
    DAMAGE_TYPES = [
        "scratches", "noise", "blur", "aging", "water_damage",
        "fire_damage", "wear_and_tear", "mold_foxing", "light_leak", "crease",
    ]
    NUM_DAMAGE_TYPES = len(DAMAGE_TYPES)
    DEGRADATION_PROB = 0.4
    MAX_DAMAGES_PER_IMAGE = 3
    CLEAN_RATIO = 0.3

    # ----- MODEL ARCHITECTURE -----
    ENCODER_CHANNELS = [32, 64, 128, 256]
    BOTTLENECK_CHANNELS = 512
    DROPOUT_RATE = 0.3

    # ----- TRAINING -----
    BATCH_SIZE = 16           # T4 has 16GB, can handle larger batches
    NUM_EPOCHS = 50
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5

    # Loss weights
    PIXEL_LOSS_WEIGHT = 1.0
    L2_LOSS_WEIGHT = 0.5
    CLASSIFICATION_LOSS_WEIGHT = 0.5
    PERCEPTUAL_LOSS_WEIGHT = 0.1

    # ----- REFERENCE GUIDANCE (Model 4 only) -----
    NUM_REFERENCES = 3

    # ----- MISC -----
    USE_AMP = True
    MAX_IMAGES = None
    VAL_SPLIT = 0.1
    TEST_SPLIT = 0.1
    NUM_WORKERS = 2           # Linux handles multiprocessing fine
    SEED = 42
    DEVICE = "cuda"

    LOG_EVERY = 50
    SAVE_EVERY = 5
    SAMPLE_EVERY = 1
    NUM_SAMPLES = 8

os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
print("Config ready.")

## 3. Dataset — Degradation Pipeline & Data Loading

In [ ]:
import os
import random
import json
import glob
import math

import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance
import torch
from torch.utils.data import Dataset


# =============================================================================
# NEGATIVE IMAGE FILTERING
# =============================================================================
NEGATIVE_KEYWORDS = ["negat", "négatif", "negativ", "invert", "film_neg"]

def is_negative(filepath):
    """Check if a file path suggests it's a film negative."""
    lower = filepath.lower()
    return any(kw in lower for kw in NEGATIVE_KEYWORDS)


def is_grayscale(filepath, threshold=5.0):
    """
    Detect B&W photos stored as RGB (all 3 channels nearly identical).
    Loads a small thumbnail for speed. Returns True if the image is grayscale.
    """
    try:
        img = Image.open(filepath).convert("RGB")
        img.thumbnail((64, 64))
        arr = np.array(img, dtype=np.float32)
        rg_diff = np.abs(arr[:,:,0] - arr[:,:,1]).mean()
        rb_diff = np.abs(arr[:,:,0] - arr[:,:,2]).mean()
        return rg_diff < threshold and rb_diff < threshold
    except Exception:
        return False


# =============================================================================
# RGB <-> Lab CONVERSION (pure numpy)
# =============================================================================

def rgb_to_lab(rgb_array):
    """
    Convert RGB image (0-255 uint8 or 0-1 float) to Lab color space.
    Returns L in [0, 100], a in [-128, 127], b in [-128, 127].
    """
    arr = rgb_array.astype(np.float32)
    if arr.max() > 1.0:
        arr = arr / 255.0

    # sRGB to linear RGB
    mask = arr > 0.04045
    arr[mask] = ((arr[mask] + 0.055) / 1.055) ** 2.4
    arr[~mask] = arr[~mask] / 12.92

    # Linear RGB to XYZ (D65)
    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
    x = r * 0.4124564 + g * 0.3575761 + b * 0.1804375
    y = r * 0.2126729 + g * 0.7151522 + b * 0.0721750
    z = r * 0.0193339 + g * 0.1191920 + b * 0.9503041

    x = x / 0.95047
    y = y / 1.00000
    z = z / 1.08883

    # XYZ to Lab
    def f(t):
        delta = 6.0 / 29.0
        mask = t > delta ** 3
        result = np.zeros_like(t)
        result[mask] = np.cbrt(t[mask])
        result[~mask] = t[~mask] / (3 * delta ** 2) + 4.0 / 29.0
        return result

    fx, fy, fz = f(x), f(y), f(z)
    L = 116.0 * fy - 16.0
    a = 500.0 * (fx - fy)
    b_ch = 200.0 * (fy - fz)

    return np.stack([L, a, b_ch], axis=-1)


def lab_to_rgb(lab_array):
    """
    Convert Lab image back to RGB (0-255 uint8).
    L in [0, 100], a in [-128, 127], b in [-128, 127].
    """
    L, a, b_ch = lab_array[:, :, 0], lab_array[:, :, 1], lab_array[:, :, 2]

    fy = (L + 16.0) / 116.0
    fx = a / 500.0 + fy
    fz = fy - b_ch / 200.0

    delta = 6.0 / 29.0

    def f_inv(t):
        mask = t > delta
        result = np.zeros_like(t)
        result[mask] = t[mask] ** 3
        result[~mask] = 3 * delta ** 2 * (t[~mask] - 4.0 / 29.0)
        return result

    x = 0.95047 * f_inv(fx)
    y = 1.00000 * f_inv(fy)
    z = 1.08883 * f_inv(fz)

    # XYZ to linear RGB
    r = x * 3.2404542 + y * -1.5371385 + z * -0.4985314
    g = x * -0.9692660 + y * 1.8760108 + z * 0.0415560
    b = x * 0.0556434 + y * -0.2040259 + z * 1.0572252

    # Linear RGB to sRGB
    rgb = np.stack([r, g, b], axis=-1)
    rgb = np.clip(rgb, 0, None)
    mask = rgb > 0.0031308
    rgb[mask] = 1.055 * (rgb[mask] ** (1.0 / 2.4)) - 0.055
    rgb[~mask] = 12.92 * rgb[~mask]

    rgb = np.clip(rgb * 255, 0, 255).astype(np.uint8)
    return rgb


# =============================================================================
# SYNTHETIC DEGRADATION FUNCTIONS
# =============================================================================

def add_scratches(img):
    """Add random scratch lines."""
    img = img.copy()
    draw = ImageDraw.Draw(img)
    w, h = img.size
    for _ in range(random.randint(2, 8)):
        if random.random() > 0.5:
            x1, x2 = random.randint(0, w // 3), random.randint(2 * w // 3, w)
            y1 = random.randint(0, h)
            y2 = y1 + random.randint(-h // 6, h // 6)
        else:
            y1, y2 = random.randint(0, h // 3), random.randint(2 * h // 3, h)
            x1 = random.randint(0, w)
            x2 = x1 + random.randint(-w // 6, w // 6)
        brightness = random.randint(180, 255)
        draw.line([(x1, y1), (x2, y2)], fill=(brightness,) * 3, width=random.randint(1, 2))
    return img


def add_noise(img):
    """Add Gaussian noise."""
    arr = np.array(img, dtype=np.float32)
    sigma = random.uniform(10, 35)
    noise = np.random.normal(0, sigma, arr.shape).astype(np.float32)
    return Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))


def add_blur(img):
    """Apply Gaussian blur."""
    return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.8, 3.0)))


def add_aging(img):
    """Simulate aging: yellowing, contrast loss, faded colors."""
    arr = np.array(img, dtype=np.float32)
    gray = np.mean(arr, axis=2, keepdims=True)
    arr = gray + random.uniform(0.2, 0.6) * (arr - gray)
    yellow = np.array([random.uniform(15, 35), random.uniform(10, 25), random.uniform(-10, 0)])
    arr = arr + yellow
    mean_val = arr.mean()
    arr = mean_val + random.uniform(0.5, 0.8) * (arr - mean_val)
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))


def add_water_damage(img):
    """Simulate water damage: blotchy discoloration patches."""
    arr = np.array(img, dtype=np.float32)
    h, w = arr.shape[:2]
    mask = np.zeros((h, w), dtype=np.float32)
    for _ in range(random.randint(2, 6)):
        cx, cy = random.randint(0, w), random.randint(0, h)
        radius = random.randint(min(h, w) // 8, min(h, w) // 3)
        y_c, x_c = np.ogrid[:h, :w]
        dist = np.sqrt((x_c - cx) ** 2 + (y_c - cy) ** 2)
        mask = np.maximum(mask, np.clip(1.0 - dist / radius, 0, 1))
    mask = mask[:, :, np.newaxis]
    stain = np.array([random.uniform(-30, -10), random.uniform(-20, -5), random.uniform(-40, -15)])
    intensity = random.uniform(0.3, 0.7)
    arr = arr + mask * stain * intensity * 2
    local_mean = arr.mean()
    arr = arr * (1 - mask * 0.2) + local_mean * mask * 0.2
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))


def add_fire_damage(img):
    """Simulate fire/heat damage: darkened edges, charring."""
    arr = np.array(img, dtype=np.float32)
    h, w = arr.shape[:2]
    y_c, x_c = np.ogrid[:h, :w]
    cy, cx = h / 2, w / 2
    max_dist = math.sqrt(cx ** 2 + cy ** 2)
    dist = np.sqrt((x_c - cx) ** 2 + (y_c - cy) ** 2)
    burn = (np.clip(dist / max_dist, 0, 1) ** random.uniform(1.5, 3.0))[:, :, np.newaxis]
    intensity = random.uniform(0.4, 0.8)
    burn_color = np.array([30, 15, 5], dtype=np.float32)
    arr = arr * (1 - burn * intensity) + burn_color * burn * intensity
    arr = arr + np.random.normal(0, 10, arr.shape) * burn
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))


def add_wear_and_tear(img):
    """Simulate physical wear: edge damage, corner wear."""
    img = img.copy()
    draw = ImageDraw.Draw(img)
    w, h = img.size
    edge_width = random.randint(3, max(4, min(w, h) // 15))
    for _ in range(random.randint(5, 15)):
        side = random.choice(["top", "bottom", "left", "right"])
        if side == "top":
            x1, y1 = random.randint(0, w), 0
            x2, y2 = x1 + random.randint(5, w // 4), random.randint(2, edge_width)
        elif side == "bottom":
            x1, y1 = random.randint(0, w), h - random.randint(2, edge_width)
            x2, y2 = x1 + random.randint(5, w // 4), h
        elif side == "left":
            x1, y1 = 0, random.randint(0, h)
            x2, y2 = random.randint(2, edge_width), y1 + random.randint(5, h // 4)
        else:
            x1, y1 = w - random.randint(2, edge_width), random.randint(0, h)
            x2, y2 = w, y1 + random.randint(5, h // 4)
        b = random.randint(160, 230)
        draw.rectangle([x1, y1, x2, y2], fill=(b, b, b))
    corner_size = random.randint(5, max(6, min(w, h) // 8))
    for cx, cy in [(0, 0), (w, 0), (0, h), (w, h)]:
        if random.random() > 0.5:
            for r in range(corner_size, 0, -1):
                a = int(180 + 75 * (1 - r / corner_size))
                draw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(a, a, a))
    return img


def add_mold_foxing(img):
    """Simulate mold/foxing spots."""
    img = img.copy()
    draw = ImageDraw.Draw(img)
    w, h = img.size
    for _ in range(random.randint(5, 25)):
        cx, cy = random.randint(0, w), random.randint(0, h)
        radius = random.randint(1, max(2, min(w, h) // 30))
        r, g, b = random.randint(50, 110), random.randint(30, 80), random.randint(10, 50)
        draw.ellipse([cx - radius, cy - radius, cx + radius, cy + radius], fill=(r, g, b))
    return img.filter(ImageFilter.GaussianBlur(radius=0.5))


def add_light_leak(img):
    """Simulate light leak — overexposed bright areas."""
    arr = np.array(img, dtype=np.float32)
    h, w = arr.shape[:2]
    lx, ly = random.choice([0, w]), random.choice([0, h])
    y_c, x_c = np.ogrid[:h, :w]
    dist = np.sqrt((x_c - lx) ** 2 + (y_c - ly) ** 2)
    max_dist = math.sqrt(w ** 2 + h ** 2)
    leak = (np.clip(1.0 - dist / (max_dist * random.uniform(0.3, 0.6)), 0, 1)
            ** random.uniform(1.0, 2.0))[:, :, np.newaxis]
    color = np.array([255, random.uniform(200, 240), random.uniform(150, 200)])
    intensity = random.uniform(0.2, 0.6)
    arr = arr * (1 - leak * intensity) + color * leak * intensity
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))


def add_crease(img):
    """Simulate fold/crease lines."""
    img = img.copy()
    draw = ImageDraw.Draw(img)
    w, h = img.size
    for _ in range(random.randint(1, 3)):
        if random.random() > 0.5:
            y = random.randint(h // 4, 3 * h // 4)
            b = random.randint(180, 230)
            draw.line([(0, y), (w, y)], fill=(b, b, b), width=random.randint(1, 3))
            off = random.choice([-2, 2])
            d = random.randint(40, 80)
            draw.line([(0, y + off), (w, y + off)], fill=(d, d, d), width=1)
        else:
            x = random.randint(w // 4, 3 * w // 4)
            b = random.randint(180, 230)
            draw.line([(x, 0), (x, h)], fill=(b, b, b), width=random.randint(1, 3))
            off = random.choice([-2, 2])
            d = random.randint(40, 80)
            draw.line([(x + off, 0), (x + off, h)], fill=(d, d, d), width=1)
    return img


DEGRADATION_FUNCTIONS = {
    "scratches": add_scratches,
    "noise": add_noise,
    "blur": add_blur,
    "aging": add_aging,
    "water_damage": add_water_damage,
    "fire_damage": add_fire_damage,
    "wear_and_tear": add_wear_and_tear,
    "mold_foxing": add_mold_foxing,
    "light_leak": add_light_leak,
    "crease": add_crease,
}


# =============================================================================
# DATASET CLASS
# =============================================================================

class RestorationDataset(Dataset):
    """
    Unified dataset for all model variants.

    Model 1 (RGB):   input = degraded grayscale [1,H,W], target = clean RGB [3,H,W]
    Models 2-4 (Lab): input = L channel [1,H,W], target = ab channels [2,H,W]
    Model 4 additionally returns K reference color images [K,3,H,W].
    """

    def __init__(self, image_paths, model_variant=None, image_size=None,
                 is_training=True, ref_paths=None):
        self.image_paths = image_paths
        self.model_variant = model_variant or config.MODEL_VARIANT
        self.image_size = image_size or config.IMAGE_SIZE
        self.is_training = is_training
        self.use_lab = self.model_variant >= 2
        self.ref_paths = ref_paths

    def __len__(self):
        return len(self.image_paths)

    def _load_and_crop(self, path):
        """Load image as RGB, resize so short side = image_size, then crop."""
        img = Image.open(path).convert("RGB")
        w, h = img.size
        scale = self.image_size / min(w, h)
        img = img.resize((int(w * scale), int(h * scale)), Image.BILINEAR)

        w, h = img.size
        if self.is_training:
            left = random.randint(0, max(0, w - self.image_size))
            top = random.randint(0, max(0, h - self.image_size))
        else:
            left = max(0, (w - self.image_size) // 2)
            top = max(0, (h - self.image_size) // 2)
        img = img.crop((left, top, left + self.image_size, top + self.image_size))

        if self.is_training and random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)

        if self.is_training and random.random() > 0.5:
            img = ImageEnhance.Brightness(img).enhance(random.uniform(0.9, 1.1))
            img = ImageEnhance.Contrast(img).enhance(random.uniform(0.9, 1.1))

        return img

    def _apply_degradation(self, color_img):
        """Apply random synthetic damage. Returns (degraded_image, damage_labels)."""
        damage_labels = np.zeros(config.NUM_DAMAGE_TYPES, dtype=np.float32)

        if random.random() < config.CLEAN_RATIO:
            return color_img, damage_labels

        all_indices = list(range(config.NUM_DAMAGE_TYPES))
        random.shuffle(all_indices)
        num_to_apply = random.randint(1, config.MAX_DAMAGES_PER_IMAGE)

        selected = []
        for idx in all_indices:
            if len(selected) >= num_to_apply:
                break
            if random.random() < config.DEGRADATION_PROB:
                selected.append(idx)
        if not selected:
            selected = [random.choice(all_indices)]

        degraded = color_img
        for idx in selected:
            damage_name = config.DAMAGE_TYPES[idx]
            degraded = DEGRADATION_FUNCTIONS[damage_name](degraded)
            damage_labels[idx] = 1.0

        return degraded, damage_labels

    def _prepare_rgb(self, color_img, degraded_img):
        """Model 1: degraded grayscale -> clean RGB."""
        gray = np.array(degraded_img.convert("L"), dtype=np.float32) / 255.0
        input_tensor = torch.from_numpy(gray).unsqueeze(0)

        rgb = np.array(color_img, dtype=np.float32) / 255.0
        target_tensor = torch.from_numpy(rgb).permute(2, 0, 1)

        return input_tensor, target_tensor

    def _prepare_lab(self, color_img, degraded_img):
        """Models 2-4: L channel -> ab channels."""
        lab = rgb_to_lab(np.array(color_img))

        degraded_lab = rgb_to_lab(np.array(degraded_img))
        L_input = degraded_lab[:, :, 0] / 100.0
        input_tensor = torch.from_numpy(L_input.astype(np.float32)).unsqueeze(0)

        a_norm = (lab[:, :, 1] + 128.0) / 255.0
        b_norm = (lab[:, :, 2] + 128.0) / 255.0
        target_tensor = torch.from_numpy(
            np.stack([a_norm, b_norm], axis=0).astype(np.float32)
        )

        return input_tensor, target_tensor

    def _load_references(self):
        """Model 4: load K random reference images as color tensors."""
        k = config.NUM_REFERENCES
        if self.ref_paths is None or len(self.ref_paths) == 0:
            return torch.zeros(k, 3, self.image_size, self.image_size)

        chosen = random.sample(self.ref_paths, min(k, len(self.ref_paths)))

        refs = []
        for p in chosen:
            try:
                ref_img = self._load_and_crop(p)
                arr = np.array(ref_img, dtype=np.float32) / 255.0
                refs.append(torch.from_numpy(arr).permute(2, 0, 1))
            except Exception:
                refs.append(torch.zeros(3, self.image_size, self.image_size))

        while len(refs) < k:
            refs.append(torch.zeros(3, self.image_size, self.image_size))

        return torch.stack(refs)

    def __getitem__(self, idx):
        path = self.image_paths[idx]

        try:
            color_img = self._load_and_crop(path)
        except Exception:
            color_img = Image.new("RGB", (self.image_size, self.image_size), (128, 128, 128))

        degraded_img, damage_labels = self._apply_degradation(color_img)

        if self.use_lab:
            input_tensor, target_tensor = self._prepare_lab(color_img, degraded_img)
        else:
            input_tensor, target_tensor = self._prepare_rgb(color_img, degraded_img)

        damage_labels = torch.from_numpy(damage_labels)

        if self.model_variant == 4:
            refs = self._load_references()
            return input_tensor, target_tensor, damage_labels, refs

        return input_tensor, target_tensor, damage_labels


# =============================================================================
# PATH COLLECTION
# =============================================================================

def collect_image_paths(data_root=None, subfolders=None, max_images=None):
    """Find all .jpg/.jpeg files under training folders, filtering out negatives and grayscale.
    Results are cached to Drive so the slow grayscale scan only runs once."""
    import time as _time

    data_root = data_root or config.DATA_ROOT
    subfolders = subfolders or config.TRAIN_SUBFOLDERS
    max_images = max_images or config.MAX_IMAGES

    # Cache lives next to DATA_ROOT on Drive — survives session restarts
    cache_path = os.path.join(os.path.dirname(data_root), "valid_paths_cache.json")

    # Try loading from cache
    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            cached_data = json.load(f)
        # Validate cache matches current DATA_ROOT (catches folder switches)
        if isinstance(cached_data, dict) and cached_data.get("data_root") == data_root:
            cached = cached_data["paths"]
            spot_check = cached[:10]
            if all(os.path.exists(p) for p in spot_check):
                print(f"Loaded {len(cached)} valid paths from cache (grayscale scan skipped).")
                random.shuffle(cached)
                if max_images and len(cached) > max_images:
                    cached = cached[:max_images]
                print(f"Found {len(cached)} training images")
                return cached
            else:
                print("Cache invalid (paths not found) — rescanning...")
        else:
            print("DATA_ROOT changed — rescanning and updating cache...")

    # No valid cache — do full scan
    all_paths = []
    for subfolder in subfolders:
        folder = os.path.join(data_root, subfolder)
        if not os.path.exists(folder):
            print(f"Warning: folder not found: {folder}")
            continue
        for pattern in ["**/*.jpg", "**/*.JPG", "**/*.jpeg", "**/*.JPEG"]:
            all_paths.extend(glob.glob(os.path.join(folder, pattern), recursive=True))

    all_paths = list(set(all_paths))

    before = len(all_paths)
    all_paths = [p for p in all_paths if not is_negative(p)]
    filtered_neg = before - len(all_paths)
    if filtered_neg > 0:
        print(f"Filtered out {filtered_neg} negative images")

    # Grayscale scan with progress indicator
    total = len(all_paths)
    print(f"Scanning {total} images for grayscale — this runs once and is then cached.")
    print("Progress will be printed every 500 images.")
    valid_paths = []
    start = _time.time()
    for i, p in enumerate(all_paths):
        if not is_grayscale(p):
            valid_paths.append(p)
        if (i + 1) % 500 == 0 or (i + 1) == total:
            elapsed = _time.time() - start
            rate = (i + 1) / max(elapsed, 1)
            remaining = (total - (i + 1)) / max(rate, 0.001)
            print(f"  [{i+1}/{total}] {len(valid_paths)} valid | ~{remaining/60:.1f} min remaining")

    filtered_gray = total - len(valid_paths)
    if filtered_gray > 0:
        print(f"Filtered out {filtered_gray} grayscale images from training folders")

    # Save cache to Drive (includes data_root so switching folders auto-invalidates)
    with open(cache_path, "w") as f:
        json.dump({"data_root": data_root, "paths": valid_paths}, f)
    print(f"Scan complete. Cache saved to {cache_path}")
    print("Future sessions will skip this scan entirely.")

    random.shuffle(valid_paths)
    if max_images and len(valid_paths) > max_images:
        valid_paths = valid_paths[:max_images]

    print(f"Found {len(valid_paths)} training images")
    return valid_paths


def collect_test_paths(data_root=None, subfolders=None):
    """Collect paths for test/inference images."""
    data_root = data_root or config.DATA_ROOT
    subfolders = subfolders or config.TEST_SUBFOLDERS

    all_paths = []
    for subfolder in subfolders:
        folder = os.path.join(data_root, subfolder)
        if not os.path.exists(folder):
            continue
        for pattern in ["**/*.jpg", "**/*.JPG", "**/*.jpeg", "**/*.JPEG"]:
            all_paths.extend(glob.glob(os.path.join(folder, pattern), recursive=True))

    all_paths = list(set(all_paths))
    all_paths = [p for p in all_paths if not is_negative(p)]
    print(f"Found {len(all_paths)} test images")
    return all_paths


def make_data_splits(image_paths, val_split=None, test_split=None):
    """
    Split image paths into train / val / test sets (80/10/10 by default).
    Uses a fixed seed so the split is always the same — test set is never seen during training.
    """
    val_split = val_split or config.VAL_SPLIT
    test_split = test_split or config.TEST_SPLIT
    random.shuffle(image_paths)
    n = len(image_paths)
    test_size = int(n * test_split)
    val_size = int(n * val_split)
    test_paths = image_paths[:test_size]
    val_paths  = image_paths[test_size:test_size + val_size]
    train_paths = image_paths[test_size + val_size:]
    print(f"Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")
    return train_paths, val_paths, test_paths


print("Dataset code ready.")

## 4. Model Architecture

In [ ]:
import torch
import torch.nn as nn


# =============================================================================
# BUILDING BLOCKS
# =============================================================================

class ConvBlock(nn.Module):
    """Two 3x3 convolutions with BatchNorm and ReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class EncoderBlock(nn.Module):
    """ConvBlock + MaxPool for downsampling."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = ConvBlock(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        features = self.conv(x)
        return features, self.pool(features)


class DecoderBlock(nn.Module):
    """Upsample + skip connection + ConvBlock + optional Dropout."""
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = ConvBlock(out_ch * 2, out_ch)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return self.dropout(x)


# =============================================================================
# CROSS-ATTENTION (Model 4 only)
# =============================================================================

class CrossAttention(nn.Module):
    """
    Cross-attention: target features query reference features to pull color info.
    Downsamples to 8x8 before computing attention to save VRAM.
    """
    ATTN_SIZE = 8

    def __init__(self, channels):
        super().__init__()
        self.query = nn.Conv2d(channels, channels, 1)
        self.key = nn.Conv2d(channels, channels, 1)
        self.value = nn.Conv2d(channels, channels, 1)
        self.scale = channels ** -0.5

    def forward(self, target, reference):
        B, C, H, W = target.shape
        S = self.ATTN_SIZE

        t_down = nn.functional.adaptive_avg_pool2d(target, S)
        r_down = nn.functional.adaptive_avg_pool2d(reference, S)

        q = self.query(t_down).view(B, C, -1)
        k = self.key(r_down).view(B, C, -1)
        v = self.value(r_down).view(B, C, -1)

        attn = torch.bmm(q.transpose(1, 2), k) * self.scale
        attn = torch.softmax(attn, dim=-1)
        out = torch.bmm(v, attn.transpose(1, 2)).view(B, C, S, S)

        out = nn.functional.interpolate(out, size=(H, W), mode="bilinear", align_corners=False)
        return out + target


class DecoderBlockWithCrossAttn(nn.Module):
    """Decoder block that also attends to reference features (Model 4)."""
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = ConvBlock(out_ch * 2, out_ch)
        self.cross_attn = CrossAttention(out_ch)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x, skip, ref_features=None):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        if ref_features is not None:
            x = self.cross_attn(x, ref_features)
        return self.dropout(x)


# =============================================================================
# REFERENCE ENCODER (Model 4 only)
# =============================================================================

class ReferenceEncoder(nn.Module):
    """Encodes K reference images into multi-scale features, averaged across K."""
    def __init__(self, encoder_channels=None):
        super().__init__()
        if encoder_channels is None:
            encoder_channels = config.ENCODER_CHANNELS

        self.enc1 = EncoderBlock(3, encoder_channels[0])
        self.enc2 = EncoderBlock(encoder_channels[0], encoder_channels[1])
        self.enc3 = EncoderBlock(encoder_channels[1], encoder_channels[2])
        self.enc4 = EncoderBlock(encoder_channels[2], encoder_channels[3])

    def forward(self, refs):
        """refs: [B, K, 3, H, W]. Returns list of 4 feature maps."""
        B, K, C, H, W = refs.shape
        x = refs.view(B * K, C, H, W)

        f1, x = self.enc1(x)
        f2, x = self.enc2(x)
        f3, x = self.enc3(x)
        f4, _ = self.enc4(x)

        def avg_refs(feat):
            _, c, h, w = feat.shape
            return feat.view(B, K, c, h, w).mean(dim=1)

        return [avg_refs(f1), avg_refs(f2), avg_refs(f3), avg_refs(f4)]


# =============================================================================
# UNIFIED MODEL
# =============================================================================

class RestorationModel(nn.Module):
    """
    Unified model for all 4 variants.
        Model 1: 1 ch in -> 3 ch out (grayscale -> RGB)
        Model 2: 1 ch in -> 2 ch out (L -> ab)
        Model 3: same as Model 2 (perceptual loss added in training)
        Model 4: same as Model 3 + reference branch with cross-attention
    """

    def __init__(self, model_variant=None):
        super().__init__()
        self.model_variant = model_variant or config.MODEL_VARIANT

        enc_ch = config.ENCODER_CHANNELS
        bneck = config.BOTTLENECK_CHANNELS
        drop = config.DROPOUT_RATE

        in_ch = 1
        out_ch = 3 if self.model_variant == 1 else 2

        # Encoder
        self.enc1 = EncoderBlock(in_ch, enc_ch[0])
        self.enc2 = EncoderBlock(enc_ch[0], enc_ch[1])
        self.enc3 = EncoderBlock(enc_ch[1], enc_ch[2])
        self.enc4 = EncoderBlock(enc_ch[2], enc_ch[3])

        # Bottleneck
        self.bottleneck = ConvBlock(enc_ch[3], bneck)

        # Decoder
        if self.model_variant == 4:
            self.dec4 = DecoderBlockWithCrossAttn(bneck, enc_ch[3], drop)
            self.dec3 = DecoderBlockWithCrossAttn(enc_ch[3], enc_ch[2], drop)
            self.dec2 = DecoderBlockWithCrossAttn(enc_ch[2], enc_ch[1], drop)
            self.dec1 = DecoderBlockWithCrossAttn(enc_ch[1], enc_ch[0], drop)
            self.ref_encoder = ReferenceEncoder(enc_ch)
        else:
            self.dec4 = DecoderBlock(bneck, enc_ch[3], drop)
            self.dec3 = DecoderBlock(enc_ch[3], enc_ch[2], drop)
            self.dec2 = DecoderBlock(enc_ch[2], enc_ch[1], drop)
            self.dec1 = DecoderBlock(enc_ch[1], enc_ch[0], drop)

        # Output heads
        self.color_head = nn.Sequential(
            nn.Conv2d(enc_ch[0], out_ch, kernel_size=1),
            nn.Sigmoid(),
        )

        self.damage_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(bneck, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(drop),
            nn.Linear(128, config.NUM_DAMAGE_TYPES),
        )

    def forward(self, x, refs=None):
        s1, x = self.enc1(x)
        s2, x = self.enc2(x)
        s3, x = self.enc3(x)
        s4, x = self.enc4(x)

        bn = self.bottleneck(x)

        if self.model_variant == 4 and refs is not None:
            ref_feats = self.ref_encoder(refs)
            x = self.dec4(bn, s4, ref_feats[3])
            x = self.dec3(x, s3, ref_feats[2])
            x = self.dec2(x, s2, ref_feats[1])
            x = self.dec1(x, s1, ref_feats[0])
        else:
            x = self.dec4(bn, s4)
            x = self.dec3(x, s3)
            x = self.dec2(x, s2)
            x = self.dec1(x, s1)

        color_out = self.color_head(x)
        damage_logits = self.damage_head(bn)

        return color_out, damage_logits


print("Model code ready.")

## 5. Utilities — Checkpoints, Metrics, VGG Loss

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from PIL import Image


# =============================================================================
# CHECKPOINTS
# =============================================================================

def save_checkpoint(model, optimizer, epoch, loss, path=None):
    if path is None:
        path = os.path.join(config.CHECKPOINT_DIR, f"checkpoint_epoch_{epoch}.pt")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": loss,
        "model_variant": config.MODEL_VARIANT,
    }, path)
    print(f"Checkpoint saved: {path}")


def load_checkpoint(model, optimizer=None, path=None):
    checkpoint = torch.load(path, map_location=config.DEVICE, weights_only=True)
    model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
    return checkpoint["epoch"]


# =============================================================================
# TENSOR -> IMAGE CONVERSION
# =============================================================================

def tensor_to_image(tensor, model_variant=None):
    variant = model_variant or config.MODEL_VARIANT
    t = tensor.detach().cpu().float()

    if t.dim() == 3 and t.shape[0] == 1:
        arr = (t.squeeze(0).numpy() * 255).clip(0, 255).astype(np.uint8)
        return Image.fromarray(arr, mode="L")

    if t.shape[0] == 3:
        arr = (t.permute(1, 2, 0).numpy() * 255).clip(0, 255).astype(np.uint8)
        return Image.fromarray(arr, mode="RGB")

    return None


def ab_tensor_to_image(ab_tensor, l_tensor):
    """Convert predicted ab + L to RGB PIL Image."""
    ab = ab_tensor.detach().cpu().numpy()
    L = l_tensor.detach().cpu().numpy()

    L_denorm = L[0] * 100.0
    a_denorm = ab[0] * 255.0 - 128.0
    b_denorm = ab[1] * 255.0 - 128.0

    lab_img = np.stack([L_denorm, a_denorm, b_denorm], axis=-1)
    rgb = lab_to_rgb(lab_img)
    return Image.fromarray(rgb)


# =============================================================================
# SAMPLE SAVING
# =============================================================================

def save_comparison(input_t, output_t, target_t, path, model_variant=None):
    variant = model_variant or config.MODEL_VARIANT

    in_img = tensor_to_image(input_t)
    if in_img.mode == "L":
        in_img = in_img.convert("RGB")

    if variant == 1:
        out_img = tensor_to_image(output_t, variant)
        tgt_img = tensor_to_image(target_t, variant)
    else:
        out_img = ab_tensor_to_image(output_t, input_t)
        tgt_img = ab_tensor_to_image(target_t, input_t)

    h, w = in_img.height, in_img.width
    gap = 2
    combined = Image.new("RGB", (w * 3 + gap * 2, h), color=(80, 80, 80))
    combined.paste(in_img, (0, 0))
    combined.paste(out_img, (w + gap, 0))
    combined.paste(tgt_img, (2 * (w + gap), 0))

    os.makedirs(os.path.dirname(path), exist_ok=True)
    combined.save(path)


def save_sample_grid(input_batch, output_batch, target_batch, epoch,
                     model_variant=None, output_dir=None):
    output_dir = output_dir or os.path.join(config.OUTPUT_DIR, "samples")
    os.makedirs(output_dir, exist_ok=True)
    variant = model_variant or config.MODEL_VARIANT

    n = min(len(input_batch), config.NUM_SAMPLES)
    for i in range(n):
        path = os.path.join(output_dir, f"epoch{epoch:03d}_sample{i}.png")
        save_comparison(input_batch[i], output_batch[i], target_batch[i],
                        path, variant)


# =============================================================================
# METRICS
# =============================================================================

def calculate_psnr(pred, target):
    mse = torch.mean((pred - target) ** 2).item()
    if mse == 0:
        return float("inf")
    return 10 * np.log10(1.0 / mse)


def calculate_damage_accuracy(logits, labels, threshold=0.5):
    preds = (torch.sigmoid(logits) > threshold).float()
    correct = (preds == labels).float()
    return correct.mean().item(), correct.mean(dim=0).tolist()


# =============================================================================
# VGG PERCEPTUAL LOSS (Model 3)
# =============================================================================

class VGGPerceptualLoss(nn.Module):
    """
    Compares intermediate VGG-16 features between predicted and target images.
    Encourages sharper, more vivid colors. VGG is frozen (no gradients).
    """
    def __init__(self):
        super().__init__()
        import torchvision.models as models
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        features = vgg.features

        self.slice1 = nn.Sequential(*features[:5])     # relu1_2
        self.slice2 = nn.Sequential(*features[5:10])   # relu2_2
        self.slice3 = nn.Sequential(*features[10:17])  # relu3_3

        for param in self.parameters():
            param.requires_grad = False

        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def _normalize(self, x):
        return (x - self.mean) / self.std

    def forward(self, pred, target):
        pred = self._normalize(pred)
        target = self._normalize(target)

        loss = 0.0
        x, y = pred, target
        for layer in [self.slice1, self.slice2, self.slice3]:
            x = layer(x)
            y = layer(y)
            loss += nn.functional.l1_loss(x, y)

        return loss


print("Utils code ready.")

## 6. Training Functions

In [ ]:
import time
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast


def ab_to_rgb_batch(ab_pred, l_input):
    """
    Convert predicted ab + L input to RGB for perceptual loss computation.
    ab_pred: [B, 2, H, W] in [0, 1]
    l_input: [B, 1, H, W] in [0, 1]
    Returns: [B, 3, H, W] in [0, 1]
    """
    B, _, H, W = ab_pred.shape
    results = []
    for i in range(B):
        L = l_input[i, 0].detach().cpu().numpy() * 100.0
        a = ab_pred[i, 0].detach().cpu().numpy() * 255.0 - 128.0
        b = ab_pred[i, 1].detach().cpu().numpy() * 255.0 - 128.0
        lab = np.stack([L, a, b], axis=-1)
        rgb = lab_to_rgb(lab).astype(np.float32) / 255.0
        results.append(torch.from_numpy(rgb).permute(2, 0, 1))
    return torch.stack(results).to(ab_pred.device)


def compute_loss(color_out, damage_logits, target, damage_labels,
                 model_variant, l1_fn, l2_fn, bce_fn,
                 perceptual_fn=None, l_input=None):
    """
    Compute the total loss for any model variant.
    Model 1: L1(RGB) + BCE(damage)
    Model 2: L1(ab) + L2(ab) + BCE(damage)
    Model 3: L1(ab) + L2(ab) + VGG_perceptual + BCE(damage)
    Model 4: L1(ab) + L2(ab) + BCE(damage)
    """
    cls_loss = bce_fn(damage_logits, damage_labels)

    if model_variant == 1:
        pixel_loss = l1_fn(color_out, target)
        total = (config.PIXEL_LOSS_WEIGHT * pixel_loss +
                 config.CLASSIFICATION_LOSS_WEIGHT * cls_loss)
        return total, pixel_loss.item(), cls_loss.item(), 0.0

    else:
        l1_loss = l1_fn(color_out, target)
        l2_loss = l2_fn(color_out, target)
        pixel_loss = config.PIXEL_LOSS_WEIGHT * l1_loss + config.L2_LOSS_WEIGHT * l2_loss

        perc_loss_val = 0.0

        if model_variant == 3 and perceptual_fn is not None and l_input is not None:
            pred_rgb = ab_to_rgb_batch(color_out, l_input)
            target_rgb = ab_to_rgb_batch(target, l_input)
            perc_loss = perceptual_fn(pred_rgb, target_rgb)
            perc_loss_val = perc_loss.item()
            pixel_loss = pixel_loss + config.PERCEPTUAL_LOSS_WEIGHT * perc_loss

        total = pixel_loss + config.CLASSIFICATION_LOSS_WEIGHT * cls_loss
        return total, l1_loss.item(), cls_loss.item(), perc_loss_val


def train_one_epoch(model, loader, optimizer, scaler, device, epoch,
                    model_variant, l1_fn, l2_fn, bce_fn, perceptual_fn):
    model.train()
    totals = {"loss": 0, "pixel": 0, "cls": 0, "perc": 0}
    n = 0

    for batch_idx, batch in enumerate(loader):
        if model_variant == 4:
            degraded, target, damage_labels, refs = batch
            refs = refs.to(device)
        else:
            degraded, target, damage_labels = batch
            refs = None

        degraded = degraded.to(device)
        target = target.to(device)
        damage_labels = damage_labels.to(device)

        optimizer.zero_grad()

        with autocast(device.type, enabled=config.USE_AMP):
            color_out, damage_logits = model(degraded, refs)
            loss, px, cl, pc = compute_loss(
                color_out, damage_logits, target, damage_labels,
                model_variant, l1_fn, l2_fn, bce_fn, perceptual_fn, degraded
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        totals["loss"] += loss.item()
        totals["pixel"] += px
        totals["cls"] += cl
        totals["perc"] += pc
        n += 1

        if (batch_idx + 1) % config.LOG_EVERY == 1 or batch_idx == 0:
            print(f"  Epoch {epoch} [{batch_idx+1}/{len(loader)}] "
                  f"Loss: {loss.item():.4f} (pixel: {px:.4f}, cls: {cl:.4f}"
                  f"{f', perc: {pc:.4f}' if model_variant >= 3 else ''})")

    return {k: v / n for k, v in totals.items()}


def validate(model, loader, device, model_variant, l1_fn, l2_fn, bce_fn, perceptual_fn):
    model.eval()
    totals = {"loss": 0, "psnr": 0, "cls_acc": 0}
    n = 0
    sample_in = sample_out = sample_tgt = None

    with torch.no_grad():
        for batch in loader:
            if model_variant == 4:
                degraded, target, damage_labels, refs = batch
                refs = refs.to(device)
            else:
                degraded, target, damage_labels = batch
                refs = None

            degraded = degraded.to(device)
            target = target.to(device)
            damage_labels = damage_labels.to(device)

            with autocast(device.type, enabled=config.USE_AMP):
                color_out, damage_logits = model(degraded, refs)
                loss, _, _, _ = compute_loss(
                    color_out, damage_logits, target, damage_labels,
                    model_variant, l1_fn, l2_fn, bce_fn, perceptual_fn, degraded
                )

            totals["loss"] += loss.item()
            totals["psnr"] += calculate_psnr(color_out, target)
            acc, _ = calculate_damage_accuracy(damage_logits, damage_labels)
            totals["cls_acc"] += acc
            n += 1

            if sample_in is None:
                sample_in = degraded.cpu()
                sample_out = color_out.cpu()
                sample_tgt = target.cpu()

    return {k: v / n for k, v in totals.items()}, sample_in, sample_out, sample_tgt


print("Training functions ready.")

## 7. Train a Model

Change `MODEL_TO_TRAIN` below to train each variant (1–4).

For a **quick test**, set `MAX_IMAGES = 200` and `NUM_EPOCHS = 2`.

For **full training**, set `MAX_IMAGES = None` and `NUM_EPOCHS = 50`.

In [ ]:
# ========== CHANGE THESE ==========
MODEL_TO_TRAIN = 1         # 1, 2, 3, or 4
NUM_EPOCHS = 50            # 2 for quick test, 50 for full
MAX_IMAGES = None          # None = all images, or e.g. 200 for quick test
BATCH_SIZE = 16            # reduce to 8 for Model 4 if OOM
RESUME_FROM = None         # set to a specific path to override auto-resume
# ==================================

# --- Auto-resume: find the latest checkpoint for this model ---
def find_latest_checkpoint(variant):
    """Scan checkpoint dir for model{N}_epoch_*.pt and return the latest one."""
    import glob, re
    pattern = os.path.join(config.CHECKPOINT_DIR, f"model{variant}_epoch_*.pt")
    files = glob.glob(pattern)
    if not files:
        return None
    def epoch_num(path):
        m = re.search(r"_epoch_(\d+)\.pt$", path)
        return int(m.group(1)) if m else -1
    latest = max(files, key=epoch_num)
    return latest

if RESUME_FROM is None:
    RESUME_FROM = find_latest_checkpoint(MODEL_TO_TRAIN)
    if RESUME_FROM:
        print(f"Auto-resume: found checkpoint {RESUME_FROM}")
    else:
        print("No existing checkpoint found — starting from scratch.")
else:
    print(f"Manual resume: {RESUME_FROM}")

# Apply settings
config.MODEL_VARIANT = MODEL_TO_TRAIN
config.MAX_IMAGES = MAX_IMAGES
config.BATCH_SIZE = BATCH_SIZE
model_variant = MODEL_TO_TRAIN

# Seed for reproducibility
random.seed(config.SEED)
np.random.seed(config.SEED)
torch.manual_seed(config.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.SEED)

device = torch.device(config.DEVICE if torch.cuda.is_available() else "cpu")

model_names = {1: "RGB Baseline", 2: "Lab Color Space",
               3: "Lab + Perceptual Loss", 4: "Reference-Guided"}
print(f"=== Training Model {model_variant}: {model_names[model_variant]} ===")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# --- Data ---
print("--- Loading Data ---")
all_paths = collect_image_paths(max_images=MAX_IMAGES)
train_paths, val_paths, test_paths = make_data_splits(all_paths)

# Save test paths so evaluation cell uses the exact same held-out set
import json as _json
_test_path_file = os.path.join(config.OUTPUT_DIR, "test_paths.json")
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
with open(_test_path_file, "w") as _f:
    _json.dump(test_paths, _f)
print(f"Test set paths saved to {_test_path_file}")

ref_paths = train_paths if model_variant == 4 else None

train_dataset = RestorationDataset(
    train_paths, model_variant=model_variant, is_training=True, ref_paths=ref_paths
)
val_dataset = RestorationDataset(
    val_paths, model_variant=model_variant, is_training=False, ref_paths=ref_paths
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=config.NUM_WORKERS, pin_memory=True,
)

# --- Model ---
print("--- Building Model ---")
model = RestorationModel(model_variant=model_variant).to(device)
params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {params:,}")

# --- Loss ---
l1_fn = nn.L1Loss()
l2_fn = nn.MSELoss()
bce_fn = nn.BCEWithLogitsLoss()

perceptual_fn = None
if model_variant == 3:
    print("Loading VGG-16 for perceptual loss...")
    perceptual_fn = VGGPerceptualLoss().to(device)

# --- Optimizer ---
optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE,
                              weight_decay=config.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)
scaler = GradScaler("cuda", enabled=config.USE_AMP)

start_epoch = 1
if RESUME_FROM:
    start_epoch = load_checkpoint(model, optimizer, RESUME_FROM) + 1

# --- Training Loop ---
print(f"
--- Starting Training ({NUM_EPOCHS} epochs, from epoch {start_epoch}) ---
")
best_val_loss = float("inf")
ckpt_prefix = f"model{model_variant}"

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    t0 = time.time()

    train_stats = train_one_epoch(
        model, train_loader, optimizer, scaler, device, epoch,
        model_variant, l1_fn, l2_fn, bce_fn, perceptual_fn
    )

    val_stats, s_in, s_out, s_tgt = validate(
        model, val_loader, device, model_variant,
        l1_fn, l2_fn, bce_fn, perceptual_fn
    )

    scheduler.step(val_stats["loss"])
    dt = time.time() - t0

    perc_str = f", perc: {train_stats['perc']:.4f}" if model_variant >= 3 else ""
    print(f"
Epoch {epoch}/{NUM_EPOCHS} ({dt:.1f}s)")
    print(f"  Train Loss: {train_stats['loss']:.4f} "
          f"(pixel: {train_stats['pixel']:.4f}, cls: {train_stats['cls']:.4f}{perc_str})")
    print(f"  Val Loss:   {val_stats['loss']:.4f} | "
          f"PSNR: {val_stats['psnr']:.2f} dB | "
          f"Damage Acc: {val_stats['cls_acc']:.2%}
")
    if epoch == start_epoch:
        remaining_epochs = NUM_EPOCHS - epoch
        est_total = dt * remaining_epochs
        print(f"  Time estimate: {dt:.0f}s/epoch x {remaining_epochs} remaining = ~{est_total/3600:.1f}h total remaining")

    # Save samples
    if epoch % config.SAMPLE_EVERY == 0 and s_in is not None:
        save_sample_grid(s_in, s_out, s_tgt, epoch, model_variant)

    # Save checkpoint
    if epoch % config.SAVE_EVERY == 0:
        save_checkpoint(model, optimizer, epoch, val_stats["loss"],
                        os.path.join(config.CHECKPOINT_DIR,
                                     f"{ckpt_prefix}_epoch_{epoch}.pt"))

    # Save best model
    if val_stats["loss"] < best_val_loss:
        best_val_loss = val_stats["loss"]
        save_checkpoint(model, optimizer, epoch, val_stats["loss"],
                        os.path.join(config.CHECKPOINT_DIR,
                                     f"{ckpt_prefix}_best.pt"))
        print(f"  >>> New best model! Val loss: {val_stats['loss']:.4f}")

print(f"
=== Training Complete ===")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Best checkpoint: {config.CHECKPOINT_DIR}/{ckpt_prefix}_best.pt")

## 8. Evaluate on Validation Set

In [ ]:
# Evaluate the model that was just trained (or load a different one)
EVAL_MODEL = MODEL_TO_TRAIN if "MODEL_TO_TRAIN" in dir() else 1  # change if needed
EVAL_CHECKPOINT = None       # None = use best checkpoint

config.MODEL_VARIANT = EVAL_MODEL

if EVAL_CHECKPOINT is None:
    EVAL_CHECKPOINT = os.path.join(config.CHECKPOINT_DIR, f"model{EVAL_MODEL}_best.pt")

print(f"--- Evaluating Model {EVAL_MODEL} on TEST SET ---")
print(f"Checkpoint: {EVAL_CHECKPOINT}")

eval_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eval_model = RestorationModel(model_variant=EVAL_MODEL).to(eval_device)
checkpoint = torch.load(EVAL_CHECKPOINT, map_location=eval_device, weights_only=True)
eval_model.load_state_dict(checkpoint["model_state_dict"])
eval_model.eval()
print(f"Loaded from epoch {checkpoint['epoch']}")

# Load the held-out test set (same paths saved during training)
import json as _json
_test_path_file = os.path.join(config.OUTPUT_DIR, "test_paths.json")
if os.path.exists(_test_path_file):
    with open(_test_path_file) as _f:
        test_paths = _json.load(_f)
    print(f"Loaded {len(test_paths)} held-out test images from {_test_path_file}")
else:
    # Fallback: re-derive the split with the same seed (will match training split)
    print("WARNING: test_paths.json not found — re-deriving split with fixed seed.")
    print("Make sure config.SEED has not changed since training.")
    all_paths = collect_image_paths()
    _, _, test_paths = make_data_splits(all_paths)

ref_paths_eval = test_paths if EVAL_MODEL == 4 else None
test_dataset = RestorationDataset(test_paths, model_variant=EVAL_MODEL,
                                   is_training=False, ref_paths=ref_paths_eval)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE,
                          shuffle=False, num_workers=config.NUM_WORKERS)

total_psnr = 0
total_acc = 0
per_type = np.zeros(config.NUM_DAMAGE_TYPES)
n = 0

with torch.no_grad():
    for batch in test_loader:
        if EVAL_MODEL == 4:
            degraded, target, damage_labels, refs = batch
            refs = refs.to(eval_device)
        else:
            degraded, target, damage_labels = batch
            refs = None

        degraded = degraded.to(eval_device)
        target = target.to(eval_device)
        damage_labels = damage_labels.to(eval_device)

        with autocast(eval_device.type, enabled=config.USE_AMP):
            color_out, damage_logits = eval_model(degraded, refs)

        total_psnr += calculate_psnr(color_out, target)
        acc, type_accs = calculate_damage_accuracy(damage_logits, damage_labels)
        total_acc += acc
        per_type += np.array(type_accs)
        n += 1

print(f"Test Set Results ({len(test_paths)} images):")
print(f"  Average PSNR:            {total_psnr / n:.2f} dB")
print(f"  Overall Damage Accuracy: {total_acc / n:.2%}")
print(f"Per-type accuracy:")
for i, name in enumerate(config.DAMAGE_TYPES):
    print(f"    {name:15s}: {per_type[i] / n:.2%}")

## 9. Restore a Test Image

In [ ]:
import matplotlib.pyplot as plt

def restore_and_show(image_path, model_variant_to_use, checkpoint_path=None):
    """Restore a single image and display the result inline."""
    config.MODEL_VARIANT = model_variant_to_use
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if checkpoint_path is None:
        checkpoint_path = os.path.join(config.CHECKPOINT_DIR,
                                        f"model{model_variant_to_use}_best.pt")

    mdl = RestorationModel(model_variant=model_variant_to_use).to(dev)
    ckpt = torch.load(checkpoint_path, map_location=dev, weights_only=True)
    mdl.load_state_dict(ckpt["model_state_dict"])
    mdl.eval()

    # Load and preprocess
    original = Image.open(image_path).convert("RGB")
    target_size = config.IMAGE_SIZE
    w, h = original.size
    scale = target_size / max(w, h)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = original.resize((new_w, new_h), Image.BILINEAR)

    padded_rgb = Image.new("RGB", (target_size, target_size), (128, 128, 128))
    left = (target_size - new_w) // 2
    top = (target_size - new_h) // 2
    padded_rgb.paste(resized, (left, top))

    if model_variant_to_use == 1:
        gray = padded_rgb.convert("L")
        arr = np.array(gray, dtype=np.float32) / 255.0
        input_tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)
    else:
        lab = rgb_to_lab(np.array(padded_rgb))
        L = lab[:, :, 0] / 100.0
        input_tensor = torch.from_numpy(L.astype(np.float32)).unsqueeze(0).unsqueeze(0)

    input_tensor = input_tensor.to(dev)

    with torch.no_grad():
        with autocast(dev.type, enabled=config.USE_AMP):
            color_out, damage_logits = mdl(input_tensor)

    # Damage detection
    probs = torch.sigmoid(damage_logits).squeeze().cpu().numpy()
    print(f"Detected damage:")
    for i, name in enumerate(config.DAMAGE_TYPES):
        marker = " <<<" if probs[i] > 0.5 else ""
        print(f"  {name:15s}: {probs[i]:.1%}{marker}")

    # Convert output to image
    if model_variant_to_use == 1:
        result_img = tensor_to_image(color_out.squeeze(0), model_variant_to_use)
    else:
        result_img = ab_tensor_to_image(color_out.squeeze(0), input_tensor.squeeze(0))

    # Display
    gray_rgb = padded_rgb.convert("L").convert("RGB")
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(gray_rgb); axes[0].set_title("Input (grayscale)"); axes[0].axis("off")
    axes[1].imshow(result_img); axes[1].set_title(f"Model {model_variant_to_use} output"); axes[1].axis("off")
    axes[2].imshow(padded_rgb); axes[2].set_title("Original"); axes[2].axis("off")
    plt.tight_layout()
    plt.show()

    return result_img

In [ ]:
# Example: restore a test image
# Uncomment and change the path to test:

# test_paths = collect_test_paths()
# if test_paths:
#     result = restore_and_show(test_paths[0], model_variant_to_use=1)

## 10. Compare All Models on Same Image

In [ ]:
def compare_all_models(image_path):
    """Run all 4 models on the same image and display side by side."""
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load and preprocess once
    original = Image.open(image_path).convert("RGB")
    target_size = config.IMAGE_SIZE
    w, h = original.size
    scale = target_size / max(w, h)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = original.resize((new_w, new_h), Image.BILINEAR)
    padded_rgb = Image.new("RGB", (target_size, target_size), (128, 128, 128))
    left = (target_size - new_w) // 2
    top = (target_size - new_h) // 2
    padded_rgb.paste(resized, (left, top))

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    gray_rgb = padded_rgb.convert("L").convert("RGB")
    axes[0].imshow(gray_rgb)
    axes[0].set_title("Input")
    axes[0].axis("off")

    for i, variant in enumerate([1, 2, 3, 4], start=1):
        ckpt_path = os.path.join(config.CHECKPOINT_DIR, f"model{variant}_best.pt")
        if not os.path.exists(ckpt_path):
            axes[i].text(0.5, 0.5, f"Model {variant}\nnot trained",
                         ha='center', va='center', transform=axes[i].transAxes)
            axes[i].axis("off")
            continue

        config.MODEL_VARIANT = variant
        mdl = RestorationModel(model_variant=variant).to(dev)
        ckpt = torch.load(ckpt_path, map_location=dev, weights_only=True)
        mdl.load_state_dict(ckpt["model_state_dict"])
        mdl.eval()

        if variant == 1:
            gray = padded_rgb.convert("L")
            arr = np.array(gray, dtype=np.float32) / 255.0
            inp = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).to(dev)
        else:
            lab = rgb_to_lab(np.array(padded_rgb))
            L = lab[:, :, 0] / 100.0
            inp = torch.from_numpy(L.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(dev)

        with torch.no_grad():
            with autocast(dev.type, enabled=config.USE_AMP):
                color_out, _ = mdl(inp)

        if variant == 1:
            result = tensor_to_image(color_out.squeeze(0), variant)
        else:
            result = ab_tensor_to_image(color_out.squeeze(0), inp.squeeze(0))

        axes[i].imshow(result)
        axes[i].set_title(f"Model {variant}")
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()


# Example usage (uncomment after training all models):
# test_paths = collect_test_paths()
# if test_paths:
#     compare_all_models(test_paths[0])